In [2]:
# En Python rápido
import pandas as pd
df = pd.read_csv("C:\\Users\\rns_2\\Documents\\Desafio-Data\\recomendador\\datos_ode_enriquecidos\\restaurantes_2026-06-02.csv")
print(df.columns.tolist())

['fuente', 'external_id', 'nombre', 'descripcion', 'categoria', 'tipo_plantilla', 'municipio', 'territorio', 'direccion', 'lat', 'lng', 'telefono', 'email', 'website', 'es_lluvia', 'es_carrito', 'es_cambiador', 'es_silla_ruedas', 'es_mascotas', 'edad_minima']


In [1]:
import pandas as pd
from pathlib import Path

# Ajusta esta ruta a donde tienes los CSV
CARPETA = r"C:\\Users\\rns_2\\Documents\\Desafio-Data\\recomendador\\datos_ode_enriquecidos"

CSV_FILES = [
    "centros-comerciales_2026-06-02.csv",
    "comercios_ode_completo_2026-06-02.csv",
    "espacios-naturales_2026-06-02.csv",
    "kulturklik_2026-06-02.csv",
    "museos_2026-06-02.csv",
    "parques_euskadi_2026-06-02.csv",
    "ocio_2026-06-02.csv",
    "planes_2026-06-02.csv",
    "recursos-culturales_2026-06-02.csv",
    "restaurantes_2026-06-02.csv",
    "rutas_2026-06-02.csv",
]

dfs = []
for filename in CSV_FILES:
    path = Path(CARPETA) / filename
    try:
        df = pd.read_csv(path, encoding="utf-8")
        df["fuente_archivo"] = filename  # para saber de dónde viene cada registro
        dfs.append(df)
        print(f"  ✓ {filename:<45} {len(df):>5} registros")
    except Exception as e:
        print(f"  ✗ {filename} — ERROR: {e}")

pool = pd.concat(dfs, ignore_index=True)

print(f"\nTotal registros en el pool : {len(pool)}")
print(f"Columnas                   : {pool.columns.tolist()}")

  ✓ centros-comerciales_2026-06-02.csv               19 registros
  ✓ comercios_ode_completo_2026-06-02.csv          1827 registros
  ✓ espacios-naturales_2026-06-02.csv                84 registros
  ✓ kulturklik_2026-06-02.csv                      2447 registros
  ✓ museos_2026-06-02.csv                           151 registros
  ✓ parques_euskadi_2026-06-02.csv                   45 registros
  ✓ ocio_2026-06-02.csv                              17 registros
  ✓ planes_2026-06-02.csv                           340 registros
  ✓ recursos-culturales_2026-06-02.csv              650 registros
  ✓ restaurantes_2026-06-02.csv                     603 registros
  ✓ rutas_2026-06-02.csv                            114 registros

Total registros en el pool : 6297
Columnas                   : ['fuente', 'external_id', 'nombre', 'descripcion', 'categoria', 'tipo_plantilla', 'municipio', 'territorio', 'direccion', 'lat', 'lng', 'telefono', 'email', 'website', 'es_interior', 'es_carrito', 'es_cambiador

In [2]:
def construir_texto(row):
    partes = [
        str(row.get("nombre",         "") or ""),
        str(row.get("categoria",      "") or ""),
        str(row.get("tipo_plantilla", "") or ""),
        str(row.get("descripcion",    "") or ""),
        str(row.get("municipio",      "") or ""),
        str(row.get("territorio",     "") or ""),
    ]
    return ". ".join(p.strip() for p in partes if p.strip())

In [4]:
# Muestra estratificada — proporcional por fuente_archivo
MUESTRA_POR_FUENTE = 20  # 10 de cada CSV = 100 registros variados

muestra_pool = (
    pool
    .dropna(subset=["nombre"])
    .groupby("fuente_archivo", group_keys=False)
    .apply(lambda x: x.sample(min(len(x), MUESTRA_POR_FUENTE), random_state=42))
    .reset_index(drop=True)
)

# Construir texto de embedding para cada registro
muestra_pool["texto_embedding"] = muestra_pool.apply(construir_texto, axis=1)

print(f"Total muestra : {len(muestra_pool)}")
print(f"\nDistribución por fuente:")
print(muestra_pool["fuente_archivo"].value_counts().to_string())
print(f"\nEjemplos de textos generados:")
for _, row in muestra_pool.head(3).iterrows():
    print(f"\n  [{row['fuente_archivo']}]")
    print(f"  {row['texto_embedding'][:120]}")

Total muestra : 216

Distribución por fuente:
fuente_archivo
comercios_ode_completo_2026-06-02.csv    20
espacios-naturales_2026-06-02.csv        20
kulturklik_2026-06-02.csv                20
restaurantes_2026-06-02.csv              20
museos_2026-06-02.csv                    20
parques_euskadi_2026-06-02.csv           20
planes_2026-06-02.csv                    20
rutas_2026-06-02.csv                     20
recursos-culturales_2026-06-02.csv       20
centros-comerciales_2026-06-02.csv       19
ocio_2026-06-02.csv                      17

Ejemplos de textos generados:

  [centros-comerciales_2026-06-02.csv]
  Dendaraba. comercio. Centros comerciales. nan. Vitoria-Gasteiz. araba

  [centros-comerciales_2026-06-02.csv]
  Bidarte. comercio. Centros comerciales. nan. Bilbao. bizkaia

  [centros-comerciales_2026-06-02.csv]
  Garbera. comercio. Centros comerciales. nan. Donostia / San Sebastián. gipuzkoa


C:\Users\rns_2\AppData\Local\Temp\ipykernel_29360\3629971069.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), MUESTRA_POR_FUENTE), random_state=42))


In [ ]:
#pip install sentence-transformers

   ---------------------------------------- 0.0/588.9 kB ? eta -:--:--
   ---- ----------------------------------- 61.4/588.9 kB 1.7 MB/s eta 0:00:01
   -------- ------------------------------- 122.9/588.9 kB 1.2 MB/s eta 0:00:01
   ----------- ---------------------------- 174.1/588.9 kB 1.3 MB/s eta 0:00:01
   ---------------- ----------------------- 245.8/588.9 kB 1.4 MB/s eta 0:00:01
   -------------------- ------------------- 307.2/588.9 kB 1.4 MB/s eta 0:00:01
   -------------------------- ------------- 389.1/588.9 kB 1.3 MB/s eta 0:00:01
   ------------------------------ --------- 450.6/588.9 kB 1.4 MB/s eta 0:00:01
   ----------------------------------- ---- 522.2/588.9 kB 1.4 MB/s eta 0:00:01
   ---------------------------------------- 588.9/588.9 kB 1.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/671.5 kB ? eta -:--:--
   --- ------------------------------------ 61.4/671.5 kB 3.4 MB/s eta 0:00:01
   -------- ------------------------------- 143.4/671.5 kB 1


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
from sentence_transformers import SentenceTransformer
import numpy as np
import time

MODELO_ST   = "intfloat/multilingual-e5-large"
TOP_K       = 10
CONSULTA    = "planes para niños de hasta 3 años en Bilbao, de interior, accesible con carrito, gratis. Preferencias museos restaurantes. Pastor por un día en Urkiola, 1 estrella; Centro comercial bidarte, 5 estrellas; restaurante asador el molino, 4 estrellas; Museo Bellas Artes, 5 estrellas."

print(f"Cargando modelo {MODELO_ST}...")
model_st = SentenceTransformer(MODELO_ST)
print("Modelo cargado.\n")

# -- Vectorizar consulta --
emb_consulta = model_st.encode(CONSULTA, normalize_embeddings=True)
print(f"Dimensión del vector : {len(emb_consulta)}")

# -- Vectorizar pool completo en batch (mucho más rápido que uno a uno) --
print(f"Vectorizando {len(muestra_pool)} registros...")
t0     = time.perf_counter()
textos = muestra_pool["texto_embedding"].tolist()
embeds = model_st.encode(textos, normalize_embeddings=True, show_progress_bar=True)
t1     = time.perf_counter()

tiempo_total = (t1 - t0) * 1000
tiempo_medio = tiempo_total / len(textos)
print(f"\nTiempo total  : {tiempo_total/1000:.1f} s")
print(f"Tiempo medio  : {tiempo_medio:.1f} ms/texto")
print(f"Estimación 1.000 reg : {tiempo_medio * 1000 / 1000:.0f} s")

# -- Similitud coseno --
scores = []
for i, row in muestra_pool.iterrows():
    score = float(np.dot(emb_consulta, embeds[muestra_pool.index.get_loc(i)]))
    scores.append({
        "nombre":   row.get("nombre", "sin nombre"),
        "categoria": row.get("categoria", ""),
        "municipio": row.get("municipio", ""),
        "fuente":   row["fuente_archivo"].replace("_2026-06-02.csv", ""),
        "score":    score,
    })

scores.sort(key=lambda x: x["score"], reverse=True)

print(f"\nTop {TOP_K} resultados para: '{CONSULTA}'\n")
for i, r in enumerate(scores[:TOP_K], 1):
    barra = "█" * int(r["score"] * 20)
    print(f"  {i:>2}. {r['score']:.3f} {barra:<15} [{r['fuente']:<22}] {r['nombre']} ({r['municipio']})")

all_scores = [r["score"] for r in scores]
print(f"\nDistribución de scores:")
print(f"  Media : {np.mean(all_scores):.3f}")
print(f"  Std   : {np.std(all_scores):.3f}")
print(f"  Min   : {np.min(all_scores):.3f}")
print(f"  Max   : {np.max(all_scores):.3f}")

c:\Users\rns_2\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Cargando modelo intfloat/multilingual-e5-large...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3305.26it/s]


Modelo cargado.

Dimensión del vector : 1024
Vectorizando 216 registros...


Batches: 100%|██████████| 7/7 [01:12<00:00, 10.42s/it]


Tiempo total  : 73.0 s
Tiempo medio  : 337.8 ms/texto
Estimación 1.000 reg : 338 s

Top 10 resultados para: 'planes para niños de hasta 3 años en Bilbao, de interior, accesible con carrito, gratis. Preferencias museos restaurantes. Pastor por un día en Urkiola, 1 estrella; Centro comercial bidarte, 5 estrellas; restaurante asador el molino, 4 estrellas; Museo Bellas Artes, 5 estrellas.'

   1. 0.864 █████████████████ [planes                ] Bilbao gourmet & Museo Bellas Artes (Bilbao)
   2. 0.844 ████████████████ [planes                ] Gastronomía y arte: tour accesible (Vitoria-Gasteiz)
   3. 0.843 ████████████████ [restaurantes          ] Islares Restaurant (Bilbao)
   4. 0.843 ████████████████ [planes                ] Dos parques naturales (Abadiño;Zeanuri)
   5. 0.838 ████████████████ [espacios-naturales    ] Parque Natural de Urkiola (Abadiño;Aramaio;Dima;Durango;Mañaria;Atxondo;Izurtza)
   6. 0.837 ████████████████ [restaurantes          ] Yandiola (Bilbao)
   7. 0.836 ██████

In [ ]:
import numpy as np
from datetime import datetime
from sentence_transformers import SentenceTransformer

# ──────────────────────────────────────────────────────────────
# CONFIGURACIÓN
# ──────────────────────────────────────────────────────────────

MODELO_ST = "intfloat/multilingual-e5-large"
TOP_K     = 10
HOY       = datetime.today()

# ──────────────────────────────────────────────────────────────
# USUARIOS HARDCODEADOS
# ──────────────────────────────────────────────────────────────

usuarios = {
    "user_1": {
        "descripcion": "padre y madre con dos niños de 1 y 3 años y perro",
        "preferencias": "montaña, museos, naturaleza, actividades en familia, paseos con perro",
        "consulta":     "planes para niños pequeños en familia con perro, naturaleza o museos",
        "historial": [
            {"nombre": "Guggenheim",     "rating": 5, "fecha": "2026-04-10"},
            {"nombre": "Playa de Ereaga","rating": 4, "fecha": "2026-05-01"},
            {"nombre": "Karting Bilbao", "rating": 2, "fecha": "2026-05-20"},
        ]
    },
    "user_2": {
        "descripcion": "padre con un niño de 2 años",
        "preferencias": "restaurantes, teatro, gastronomía, cultura, espectáculos",
        "consulta":     "planes con niño de 2 años, comer bien o teatro y cultura",
        "historial": [
            {"nombre": "Aizian",         "rating": 5, "fecha": "2026-05-15"},
            {"nombre": "Teatro Arriaga", "rating": 5, "fecha": "2026-03-10"},
        ]
    }
}

# ──────────────────────────────────────────────────────────────
# PLANES PROMOCIONADOS (simulados)
# ──────────────────────────────────────────────────────────────

promocionados = {
    "Museo Guggenheim Bilbao": 1.3,
    "Aquarium Donostia":       1.2,
    "Aizian":                  1.4,
}

# ──────────────────────────────────────────────────────────────
# HELPERS
# ──────────────────────────────────────────────────────────────

def cosine_similarity(v1, v2):
    a, b = np.array(v1), np.array(v2)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom != 0 else 0.0

def calcular_penalizacion(fecha_str: str) -> float:
    """Penalización que decae linealmente hasta 0 a los 90 días."""
    fecha  = datetime.strptime(fecha_str, "%Y-%m-%d")
    dias   = (HOY - fecha).days
    if dias >= 90:
        return 0.0
    return round(0.3 * (1 - dias / 90), 4)

def calcular_alpha_beta(n_historial: int) -> tuple:
    """α/β dinámico según riqueza del historial."""
    if n_historial < 3:
        return 0.9, 0.1
    elif n_historial <= 10:
        return 0.7, 0.3
    else:
        return 0.5, 0.5

def construir_prompt_usuario(usuario: dict) -> str:
    """
    Construye el prompt enriquecido con perfil + historial ponderado por rating.
    Usa prefijo 'query:' requerido por multilingual-e5.
    """
    historial = usuario["historial"]

    encantados  = [h["nombre"] for h in historial if h["rating"] == 5]
    gustados    = [h["nombre"] for h in historial if h["rating"] == 4]
    regulares   = [h["nombre"] for h in historial if h["rating"] == 3]
    no_gustados = [h["nombre"] for h in historial if h["rating"] <= 2]

    partes = [f"query: {usuario['consulta']}. Perfil: {usuario['descripcion']}."]

    if encantados:
        partes.append(f"Le ha encantado (5★): {', '.join(encantados)}.")
    if gustados:
        partes.append(f"Le ha gustado (4★): {', '.join(gustados)}.")
    if regulares:
        partes.append(f"Le ha parecido regular (3★): {', '.join(regulares)}.")
    if no_gustados:
        partes.append(f"No le ha gustado (1-2★): {', '.join(no_gustados)}.")

    return " ".join(partes)

def construir_texto_lugar(row) -> str:
    partes = [
        str(row.get("nombre",         "") or ""),
        str(row.get("categoria",      "") or ""),
        str(row.get("tipo_plantilla", "") or ""),
        str(row.get("descripcion",    "") or ""),
        str(row.get("municipio",      "") or ""),
        str(row.get("territorio",     "") or ""),
    ]
    return "passage: " + ". ".join(p.strip() for p in partes if p.strip())

# ──────────────────────────────────────────────────────────────
# CARGAR MODELO Y VECTORIZAR POOL
# ──────────────────────────────────────────────────────────────

print(f"Cargando modelo {MODELO_ST}...")
model_st = SentenceTransformer(MODELO_ST)
print("Modelo cargado.\n")

print(f"Vectorizando pool de {len(muestra_pool)} registros...")
textos_pool = muestra_pool.apply(construir_texto_lugar, axis=1).tolist()
embeddings_pool = model_st.encode(textos_pool, normalize_embeddings=True, show_progress_bar=True)
print("Pool vectorizado.\n")

# ──────────────────────────────────────────────────────────────
# RECOMENDADOR
# ──────────────────────────────────────────────────────────────

def recomendar(user_id: str, usuario: dict):
    historial   = usuario["historial"]
    n_historial = len(historial)
    alpha, beta = calcular_alpha_beta(n_historial)

    print(f"\n{'='*65}")
    print(f"  USUARIO: {user_id}")
    print(f"  Perfil : {usuario['descripcion']}")
    print(f"  α={alpha} β={beta} (historial: {n_historial} planes)")
    print(f"{'='*65}")

    # -- Prompt enriquecido con historial --
    prompt = construir_prompt_usuario(usuario)
    print(f"\nPrompt generado:\n  {prompt}\n")

    # -- Vector de consulta enriquecida --
    emb_consulta = model_st.encode(prompt, normalize_embeddings=True)

    # -- Vector de perfil (media ponderada del historial con rating >= 4) --
    historial_positivo = [h for h in historial if h["rating"] >= 4]
    if historial_positivo and beta > 0:
        nombres_positivos = [h["nombre"] for h in historial_positivo]
        pesos             = [h["rating"] for h in historial_positivo]
        embs_historial    = model_st.encode(
            [f"passage: {n}" for n in nombres_positivos],
            normalize_embeddings=True
        )
        emb_perfil = np.average(embs_historial, axis=0, weights=pesos)
        emb_perfil = emb_perfil / np.linalg.norm(emb_perfil)
    else:
        emb_perfil = emb_consulta  # fallback si no hay historial positivo

    # -- Penalizaciones por lugar ya visitado --
    penalizaciones = {
        h["nombre"]: calcular_penalizacion(h["fecha"])
        for h in historial
    }

    # -- Calcular score final por candidato --
    candidatos = []
    for i, row in muestra_pool.iterrows():
        idx    = muestra_pool.index.get_loc(i)
        emb    = embeddings_pool[idx]
        nombre = row.get("nombre", "sin nombre")

        sim_consulta = cosine_similarity(emb_consulta, emb)
        sim_perfil   = cosine_similarity(emb_perfil,   emb)
        score_base   = alpha * sim_consulta + beta * sim_perfil

        # Penalización temporal
        penalizacion = penalizaciones.get(nombre, 0.0)
        score_penalizado = score_base - penalizacion

        # Boost promocional
        boost       = promocionados.get(nombre, 1.0)
        score_final = score_penalizado * boost
        patrocinado = nombre in promocionados

        candidatos.append({
            "nombre":        nombre,
            "categoria":     row.get("categoria",  ""),
            "municipio":     row.get("municipio",  ""),
            "fuente":        row["fuente_archivo"].replace("_2026-06-02.csv", ""),
            "sim_consulta":  round(sim_consulta,   4),
            "sim_perfil":    round(sim_perfil,      4),
            "score_base":    round(score_base,      4),
            "penalizacion":  round(penalizacion,    4),
            "boost":         boost,
            "score_final":   round(score_final,     4),
            "patrocinado":   patrocinado,
        })

    candidatos.sort(key=lambda x: x["score_final"], reverse=True)

    # -- Aplicar regla: max 2 promocionados en top 5 --
    ranking_final  = []
    promos_en_top5 = 0

    for c in candidatos:
        posicion = len(ranking_final) + 1

        if c["patrocinado"] and posicion <= 5:
            if promos_en_top5 >= 2:
                # Mover al final del top 10
                ranking_final.append({**c, "_movido": True})
                continue
            promos_en_top5 += 1

        ranking_final.append({**c, "_movido": False})

        if len(ranking_final) >= TOP_K:
            break

    # -- Imprimir resultado --
    print(f"  {'#':<3} {'Score':>6} {'Base':>6} {'Pen':>5} {'Boost':>5} {'P':>2}  {'Nombre':<30} {'Categoría'}")
    print(f"  {'-'*3} {'-'*6} {'-'*6} {'-'*5} {'-'*5} {'-'*2}  {'-'*30} {'-'*15}")

    for i, c in enumerate(ranking_final, 1):
        promo_flag = "1-4" if c["patrocinado"] else " "
        pen_str    = f"-{c['penalizacion']}" if c["penalizacion"] > 0 else "  -  "
        print(
            f"  {i:<3} {c['score_final']:>6.3f} {c['score_base']:>6.3f} "
            f"{pen_str:>5} {c['boost']:>5.1f} {promo_flag:>2}  "
            f"{c['nombre']:<30} {c['categoria']}"
        )

    return ranking_final


# ──────────────────────────────────────────────────────────────
# EJECUTAR PARA AMBOS USUARIOS
# ──────────────────────────────────────────────────────────────

resultados = {}
for user_id, usuario in usuarios.items():
    resultados[user_id] = recomendar(user_id, usuario)

Cargando modelo intfloat/multilingual-e5-large...


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3295.15it/s]


Modelo cargado.

Vectorizando pool de 216 registros...


Batches: 100%|██████████| 7/7 [01:07<00:00,  9.65s/it]


Pool vectorizado.


  USUARIO: user_1
  Perfil : padre y madre con dos niños de 1 y 3 años y perro
  α=0.7 β=0.3 (historial: 3 planes)

Prompt generado:
  query: planes para niños pequeños en familia con perro, naturaleza o museos. Perfil: padre y madre con dos niños de 1 y 3 años y perro. Le ha encantado (5★): Guggenheim. Le ha gustado (4★): Playa de Ereaga. No le ha gustado (1-2★): Karting Bilbao.

  #    Score   Base   Pen Boost  P  Nombre                         Categoría
  --- ------ ------ ----- ----- --  ------------------------------ ---------------
  1    0.853  0.853   -     1.0     Parque Etxebarria              naturaleza
  2    0.841  0.841   -     1.0     La Ruta del Arte BBK: "Animalario" evento_cultural
  3    0.841  0.841   -     1.0     Dos parques naturales          plan
  4    0.841  0.841   -     1.0     Parque de Atxa                 naturaleza
  5    0.840  0.840   -     1.0     Kobaundi                       naturaleza
  6    0.840  0.840   -     1.0     Bilbao,